# Laboratorio 2. Complejidad y búsqueda de hiperparámetros
### ISIS2611 · Aprendizaje de Máquina | Caso AlpesPlanck

**Integrantes:** (nombres completos y códigos)

**Grupo de entrega:** GL2

**Fecha límite:** 14 de septiembre, 20:00

---

> **Cómo usar este esqueleto.** Cada sección contiene una celda de texto que explica *qué* hay que hacer,
> *por qué* se hace y *qué* se debe justificar por escrito, seguida de una celda de código vacía con
> comentarios `# TODO`. Ninguna celda trae la solución. Antes de escribir código en una sección, conviene
> poder responder en voz alta la pregunta de control que aparece al final de cada bloque.
> Al terminar, elimina las notas marcadas como *"Nota del esqueleto"*, para que el notebook entregado se
> lea como un informe técnico y no como una guía.

## 0. Mapa del laboratorio y trazabilidad con la rúbrica

Antes de programar conviene ver el laboratorio completo, porque el peso de la nota no está donde suele
estar la atención. El código de los modelos vale 40 % y el análisis escrito vale 35 %.

| Sección del notebook | Actividad del enunciado | Criterio de rúbrica | Peso |
|:---|:---|:---|:---:|
| 1. Configuración, datos y protocolo de evaluación | Preparación previa | Transversal, soporta todos | 0 % directo |
| 2. Regresión polinomial con GridSearchCV | Actividad 1 | Criterio 1 | 15 % |
| 3. Curvas de validación | Actividad 2 | Se evalúa dentro de los criterios 1 y 6 | incluido |
| 4. Ridge y Lasso | Actividad 3 | Criterio 2 | 15 % |
| 5. Polinomial regularizado | Actividad 4 | Criterio 3 | 10 % |
| 6. Tabla comparativa y selección del mejor modelo | Actividad 5 | Criterio 4 | 5 % |
| 7. Intervalos de confianza por bootstrapping | Actividad 6 | Criterio 5 | 10 % |
| 8. Análisis de resultados | Sección "Análisis de resultados" | Criterio 6 | 35 % |
| 9. Video explicativo | Actividad 7 | Criterio 7 | 5 % |
| 10. Uso de herramientas de IA generativa | Reglas del curso | Criterio 8 | 5 % |

**Nota del esqueleto (importante).** La actividad 2, las curvas de validación, no tiene una fila propia
en la rúbrica. Eso no significa que se pueda omitir: es la evidencia gráfica que sostiene varias de las
preguntas del análisis, que sí valen 35 %. Omitirla cuesta puntos por dos vías, en el criterio 1 por
justificación incompleta y en el criterio 6 por respuestas sin soporte.

**Segunda observación.** El criterio 6 vale más que cualquier modelo individual. Un notebook con cuatro
modelos impecables y respuestas de dos líneas obtiene menos nota que uno con modelos correctos y un
análisis bien argumentado. Reserva tiempo real para la sección 8.

---
## 1. Configuración, datos y protocolo de evaluación

Esta sección no aparece en la rúbrica, pero condiciona la validez de todo lo demás. Si la partición o
el esquema de validación están mal planteados, todos los números posteriores quedan invalidados aunque
el código corra sin errores.

### 1.1 Importación de librerías

**Qué hacer.** Importar en una sola celda todo lo que se usará en el notebook, agrupado por origen:
manipulación de datos, visualización, y los componentes de scikit-learn (partición, validación cruzada,
búsqueda de hiperparámetros, curvas de validación, transformadores de preprocesamiento, escaladores,
generación de características polinomiales, los tres estimadores lineales y las métricas de regresión).

**Por qué en una sola celda.** Un lector, y el profesor, debe poder reconstruir el entorno leyendo una
celda. Las importaciones dispersas a lo largo del notebook dificultan la revisión y suelen producir
errores al reejecutar desde cero.

**Qué agregar aquí también.** La fijación de una semilla global y una constante `RANDOM_STATE` que se
reutilice en cada partición, en cada `KFold` y en cada remuestreo. Sin semilla fija, los resultados
cambian entre ejecuciones y las conclusiones del análisis dejan de ser verificables.

In [ ]:
# TODO 1.1
# - Importar librerías de manipulación de datos y visualización.
# - Importar de scikit-learn: partición, validación cruzada, búsqueda en rejilla,
#   curva de validación, pipeline, transformador por columnas, imputación,
#   escaladores (tres alternativas), características polinomiales,
#   codificación de categóricas, los estimadores lineales y las métricas.
# - Definir RANDOM_STATE y fijar la semilla.
# - Configurar el formato de impresión de pandas y el estilo de las gráficas.

### 1.2 Recuperación de la preparación realizada en el Laboratorio 1

El Laboratorio 2 continúa el caso desarrollado en el Laboratorio 1. Por esta razón, no se repetirá la etapa completa de exploración de datos, sino que se reutilizarán las decisiones de limpieza y preparación que ya fueron justificadas previamente.

El archivo entregado para este laboratorio corresponde nuevamente al conjunto de datos original, por lo que es necesario reproducir las transformaciones determinísticas realizadas en el Laboratorio 1 antes de comenzar el nuevo proceso de modelado.

En particular, se recuperará la preparación correspondiente al modelo final seleccionado en el Laboratorio 1. Esta preparación incluye las correcciones de calidad realizadas, la construcción de variables temporales a partir de `fecha`, el tratamiento de la dirección del viento mediante `sector_viento` y la construcción de variables de rango para presión, humedad, viento y ráfagas.

Las operaciones que requieren aprender parámetros de los datos, como la imputación, el escalamiento y la codificación de variables categóricas, no se aplicarán todavía sobre todo el conjunto. Estas transformaciones permanecerán dentro de los pipelines para que sean ajustadas únicamente utilizando los datos de entrenamiento.

Antes de continuar se verificará que las dimensiones, las variables disponibles y la variable objetivo `temp_max_manana` correspondan con las obtenidas al finalizar la preparación del Laboratorio 1.

In [ ]:
# TODO 1.2
# - Cargar el conjunto limpio resultante del Laboratorio 1.
# - Verificar dimensiones, tipos, faltantes y descriptivos de la variable objetivo.
# - Dejar constancia escrita de qué transformaciones vienen heredadas del Lab 1.

### 1.3 Separación de características y variable objetivo

Una vez reproducida la preparación determinística realizada en el Laboratorio 1, se separará la variable objetivo `temp_max_manana` de las características predictoras.

Para mantener continuidad con el protocolo utilizado en el Laboratorio 1, se conservará la misma estrategia de partición:

- 75 % de los datos para entrenamiento.
- 25 % de los datos para test.
- `random_state = 42`.

La partición se realizará antes de cualquier transformación que requiera aprender parámetros de los datos. De esta manera, procesos como imputación, escalamiento y codificación categórica serán ajustados exclusivamente con los datos correspondientes al entrenamiento.

El conjunto de test quedará reservado durante la búsqueda de hiperparámetros y la selección de modelos. Las decisiones de modelado se tomarán utilizando únicamente el conjunto de entrenamiento y validación cruzada.

También se conservarán las decisiones tomadas en el Laboratorio 1 respecto al tratamiento de variables como `fecha`, `anio`, las variables temporales derivadas y las variables utilizadas finalmente para el modelado.

In [ ]:
# TODO 1.3
# - Definir la lista de columnas predictoras y la variable objetivo.
# - Decidir el tratamiento de la columna de fecha y documentarlo.
# - Ejecutar la partición entrenamiento / test con la semilla fija.
# - Reportar el tamaño de cada conjunto y verificar que las distribuciones de y sean comparables.

### 1.4 Protocolo de validación y métricas

La selección de modelos y de hiperparámetros se realizará mediante validación cruzada sobre el conjunto de entrenamiento.

Se definirá un único esquema de validación cruzada que será reutilizado en todas las búsquedas de hiperparámetros. Utilizar el mismo esquema permite realizar comparaciones consistentes entre los diferentes modelos construidos durante el laboratorio.

La configuración concreta del esquema de validación cruzada se definirá con base en las prácticas trabajadas en clase.

Las métricas que se utilizarán para evaluar los modelos son las solicitadas en el enunciado:

- RMSE.
- MAE.
- R².

Se definirá además una métrica principal para la selección realizada mediante `GridSearchCV`. Esta decisión se tomará antes de observar los resultados finales y deberá ser justificada.

En scikit-learn algunas métricas de error utilizadas durante la validación cruzada se representan con signo negativo debido a la convención interna de maximización. Al presentar los resultados, estas métricas se transformarán nuevamente a su interpretación habitual para evitar reportar valores negativos de error.

In [ ]:
# TODO 1.4
# - Instanciar el objeto de validación cruzada que se reutilizará en todo el notebook.
# - Definir el diccionario de scoring con RMSE, MAE y R2.
# - Declarar la métrica principal para refit y justificar la elección en texto.

### 1.5 Funciones auxiliares de reporte

**Qué hacer.** Escribir dos o tres funciones cortas que se reutilizarán en cada sección:

- Una que reciba un objeto de búsqueda ajustado y devuelva una fila con el nombre del modelo, los mejores
  hiperparámetros, y la media y la desviación estándar de las tres métricas en validación cruzada.
- Una que grafique una curva de validación recibiendo el eje del hiperparámetro y los resultados de
  entrenamiento y validación, incluyendo la banda de variabilidad.
- Opcionalmente, una que extraiga los coeficientes de un pipeline junto con los nombres de las
  características generadas.

**Por qué hacerlo ahora.** Sin estas funciones el notebook termina con cuatro bloques de código casi
idénticos, la tabla comparativa se arma a mano y aparecen inconsistencias entre secciones. Además, la
función de extracción de nombres es indispensable en la sección 4, donde hay que decir *qué* variables
anula Lasso, no solo cuántas.

---

**Pregunta de control de la sección 1.** Si escalas todo el conjunto y después partes en entrenamiento y
test, ¿qué información exactamente ha cruzado la frontera, y por qué la estimación del error resulta
optimista y no pesimista?

In [ ]:
# TODO 1.5
# - Función de resumen de resultados de una búsqueda (media y desviación por métrica).
# - Función de graficación de curvas de validación con banda de variabilidad.
# - Función de extracción de coeficientes y nombres de características desde un pipeline.

---
## 2. Actividad 1. Modelo de regresión polinomial con búsqueda de hiperparámetros
**Rúbrica: criterio 1, 15 %**

El objetivo declarado en el enunciado es analizar cómo varía el desempeño a medida que aumenta la
complejidad e identificar posibles indicios de sobreajuste. El modelo no es el fin, es el instrumento
para observar ese fenómeno.

### 2.1 Construcción del pipeline de regresión polinomial

Para estudiar el efecto de la complejidad se construirá un pipeline que combine el preprocesamiento de las variables con la generación de características polinomiales y un modelo de regresión lineal.

Como punto de partida se utilizará la representación seleccionada en el Laboratorio 1.

Después de la ingeniería de características realizada previamente, las variables numéricas consideradas son:

- `presion_media`
- `presion_desv`
- `humedad_media`
- `humedad_desv`
- `viento_media`
- `viento_desv`
- `rafaga_media`
- `rafaga_desv`
- `viento_norte`
- `viento_este`
- `registros_del_dia`
- `presion_rango`
- `humedad_rango`
- `viento_rango`
- `rafaga_rango`

Las variables categóricas utilizadas son:

- `sector_viento`
- `mes_calendario`

Las transformaciones que dependen de los datos, como imputación, escalamiento y codificación, permanecerán dentro del pipeline para evitar fuga de información durante la validación cruzada.

La expansión polinomial se aplicará sobre las variables numéricas. El grado del polinomio no se fijará manualmente, sino que será tratado como un hiperparámetro durante la búsqueda.

La estructura exacta del pipeline deberá garantizar que todas las transformaciones sean aprendidas únicamente a partir del subconjunto de entrenamiento correspondiente en cada partición de validación cruzada.

In [ ]:
# TODO 2.1
# - Identificar columnas numéricas y categóricas de forma programática.
# - Construir la rama numérica: imputación -> escalado -> expansión polinomial.
# - Construir la rama categórica: imputación -> codificación.
# - Ensamblar el ColumnTransformer y verificar que transforma sin error una muestra pequeña.

### 2.2 Definición de la búsqueda de hiperparámetros

El modelo de regresión polinomial se ajustará utilizando `GridSearchCV`.

La búsqueda incluirá los dos elementos solicitados en el enunciado:

- El grado de las características polinomiales.
- Diferentes estrategias de escalamiento dentro del pipeline.

Los valores concretos que se probarán para cada hiperparámetro se definirán de acuerdo con las estrategias trabajadas en las prácticas de clase.

Antes de ejecutar la búsqueda se revisará el número total de combinaciones y el costo computacional esperado, debido a que el número de características puede crecer rápidamente a medida que aumenta el grado del polinomio.

La búsqueda utilizará el mismo esquema de validación cruzada y las mismas métricas definidas previamente.

In [ ]:
# TODO 2.2
# - Definir el diccionario de la rejilla con el eje de grado y el eje de escalador.
# - Calcular e imprimir el número total de ajustes que implica la rejilla.

### 2.3 Búsqueda y evaluación mediante validación cruzada

Se ejecutará `GridSearchCV` sobre el pipeline construido anteriormente.

Para cada configuración se registrarán las métricas de validación cruzada:

- RMSE.
- MAE.
- R².

Además del promedio de cada métrica, se conservará su desviación estándar entre los pliegues de validación, ya que esta información permitirá analizar posteriormente la estabilidad de los modelos.

También se almacenará el desempeño sobre entrenamiento para apoyar el análisis de las curvas de validación y la identificación de posibles señales de sobreajuste.

En esta etapa no se utilizará el conjunto de test para seleccionar grados, escaladores ni ninguna otra decisión del modelo.

Al finalizar se reportarán los mejores hiperparámetros encontrados y una tabla con los resultados de las configuraciones evaluadas.

In [ ]:
# TODO 2.3
# - Instanciar y ajustar GridSearchCV con return_train_score activado.
# - Imprimir los mejores hiperparámetros.
# - Construir una tabla con todas las combinaciones, ordenada por la métrica principal,
#   con media y desviación estándar de RMSE, MAE y R2, y con el puntaje de entrenamiento.
# - Guardar el mejor estimador en una variable para la comparación de la sección 6.

### 2.4 Justificación escrita de las decisiones

**Qué escribir.** Un párrafo, no una lista de viñetas, que responda: por qué ese rango de grados, por qué
esos escaladores, por qué esa métrica principal, y qué muestra la tabla de resultados sobre la relación
entre complejidad y desempeño. La rúbrica pide el modelo "justificando las decisiones tomadas", de modo
que la mitad del criterio 1 vive en este texto y no en el código.

## 3. Actividad 2. Curvas de validación

A partir del modelo polinomial anterior se analizará cómo cambia el desempeño a medida que aumenta el grado del polinomio.

Se representará el error promedio obtenido mediante validación cruzada para los diferentes grados explorados, incluyendo su variabilidad entre los pliegues.

Adicionalmente, se puede representar el error de entrenamiento junto con el de validación. Esta comparación permite observar con mayor claridad la brecha entre ambos desempeños y facilita la identificación de posibles señales de sobreajuste.

La interpretación se realizará utilizando el trade-off sesgo-varianza:

- Un modelo demasiado simple puede presentar subajuste.
- Al aumentar la complejidad puede mejorar la capacidad de ajuste.
- Si la complejidad continúa aumentando, el desempeño sobre entrenamiento puede seguir mejorando mientras el desempeño de validación deja de mejorar o empeora.

La identificación del punto de sobreajuste se realizará únicamente a partir de los resultados obtenidos en este laboratorio, sin asumir previamente en qué grado ocurrirá.

In [ ]:
# TODO 3
# - Obtener errores de entrenamiento y validación por grado, con media y desviación.
# - Graficar ambas series con banda de variabilidad, ejes rotulados y leyenda.
# - Marcar el mínimo de validación en el gráfico.
# - Redactar debajo la interpretación en términos de sesgo, varianza y brecha.

## 4. Actividad 3. Modelos de regresión lineal regularizados

En esta sección se evaluará el efecto de la regularización sobre modelos lineales.

A diferencia de la actividad anterior, en esta etapa no se utilizará expansión polinomial. El objetivo es aislar el efecto de las penalizaciones L2 y L1 sobre una misma representación de las variables.

Se construirán tres modelos comparables:

- Regresión lineal sin regularización.
- Ridge, con penalización L2.
- Lasso, con penalización L1.

Los tres modelos utilizarán la misma representación de las variables, el mismo protocolo de validación cruzada y las mismas métricas.

### 4.1 Modelo lineal sin regularización

Primero se construirá un modelo lineal sin regularización utilizando la misma preparación de datos que se utilizará posteriormente con Ridge y Lasso.

Este modelo funcionará como referencia para poder analizar de forma aislada el efecto que produce la regularización.

La comparación deberá realizarse utilizando el mismo conjunto de variables, el mismo esquema de validación cruzada y las mismas métricas.

In [ ]:
# TODO 4.1
# - Construir el pipeline con Ridge.
# - Definir la rejilla con alpha en escala logarítmica y el eje de escalador.
# - Ejecutar la búsqueda con el mismo objeto de validación cruzada de 1.4.
# - Reportar mejores hiperparámetros y las tres métricas con media y desviación.

### 4.2 Regresión Ridge

Se construirá un pipeline de regresión Ridge incorporando la estrategia de escalamiento dentro del pipeline.

El parámetro de penalización será tratado como un hiperparámetro y será explorado mediante `GridSearchCV`, junto con las estrategias de escalamiento definidas para el laboratorio.

Para el mejor modelo se reportarán:

- Los hiperparámetros seleccionados.
- RMSE promedio y desviación estándar.
- MAE promedio y desviación estándar.
- R² promedio y desviación estándar.
- La magnitud de los coeficientes obtenidos.

Posteriormente estos resultados se compararán con los del modelo lineal sin regularización.

In [ ]:
# TODO 4.2
# - Construir el pipeline con Lasso y un número de iteraciones suficiente.
# - Definir la rejilla y ejecutar la búsqueda.
# - Verificar que no queden advertencias de convergencia sin resolver ni documentar.

### 4.3 Regresión Lasso

Se repetirá el procedimiento anterior utilizando regresión Lasso.

El parámetro de penalización se incluirá dentro de la búsqueda de hiperparámetros junto con las estrategias de escalamiento.

Además del desempeño predictivo, se estudiará el efecto de la penalización L1 sobre los coeficientes.

En particular, se identificará:

- Cuántos coeficientes quedan en cero.
- Qué características son eliminadas automáticamente.
- Qué características conservan coeficientes distintos de cero.
- Cuáles presentan mayor magnitud absoluta.

Este análisis permitirá discutir el uso de Lasso como mecanismo de selección automática de características y su efecto sobre la interpretabilidad del modelo.

In [ ]:
# TODO 4.3
# - Recorrer un rango logarítmico de alpha ajustando cada penalización.
# - Almacenar los coeficientes de cada ajuste.
# - Graficar coeficiente contra alpha, con escala logarítmica en el eje horizontal,
#   una figura por penalización.

### 4.4 Comparación entre regresión lineal, Ridge y Lasso

Se construirá una tabla comparativa utilizando únicamente resultados obtenidos mediante el mismo esquema de validación cruzada.

La tabla incluirá como mínimo:

| Modelo | Mejores hiperparámetros | RMSE CV | Desv. RMSE | MAE CV | R² CV | Coeficientes no nulos |
|:---|:---|---:|---:|---:|---:|---:|
| Regresión lineal | — | | | | | |
| Ridge | | | | | | |
| Lasso | | | | | | |

A partir de esta comparación se analizará si la regularización mejora la capacidad de generalización y cómo modifica la magnitud de los coeficientes.

En el caso de Lasso también se analizará el efecto de la selección automática de variables.

In [ ]:
# TODO 4.4
# - Extraer coeficientes y nombres de características del mejor pipeline Lasso.
# - Contar características totales y no nulas.
# - Listar las de mayor magnitud absoluta con su signo.
# - Comparar esa lista con los coeficientes de mayor magnitud en Ridge.

### 4.5 Comparación entre el modelo sin regularización, Ridge y Lasso

**Qué hacer.** Una comparación de tres filas: modelo lineal sin penalizar, Ridge y Lasso, con RMSE, MAE y
R² en validación cruzada, la desviación estándar entre pliegues y una medida de la magnitud de los
coeficientes, por ejemplo la norma o el conteo de coeficientes no nulos.

**Nota del esqueleto.** Para que la fila "sin regularización" sea comparable, debe usar el mismo
preprocesamiento y el mismo esquema de pliegues. Si comparas un modelo lineal simple contra un Ridge
sobre características polinomiales, no estás midiendo el efecto de la regularización sino el de la
expansión.

---

**Pregunta de control de la sección 4.** ¿Qué ocurre exactamente con la penalización si eliminas el
escalador del pipeline antes de regularizar? Explica el mecanismo, no solo el resultado.

In [ ]:
# TODO 4.5
# - Ajustar el modelo sin penalización con el mismo preprocesamiento y los mismos pliegues.
# - Consolidar las tres filas en una tabla con métricas, desviaciones y magnitud de coeficientes.
# - Redactar la lectura de la tabla.

## 5. Actividad 4. Regresión polinomial regularizada

En esta actividad se combinarán los dos mecanismos de control de complejidad estudiados anteriormente:

- La generación de características polinomiales.
- La regularización de los coeficientes.

Se construirá un pipeline que combine `PolynomialFeatures` con un estimador regularizado, utilizando Ridge o Lasso según la evidencia obtenida en la sección anterior.

La búsqueda mediante `GridSearchCV` explorará simultáneamente:

- El grado del polinomio.
- El parámetro de penalización.
- La estrategia de escalamiento.

El modelo será evaluado mediante RMSE, MAE y R² utilizando el mismo esquema de validación cruzada empleado en las demás actividades.

El análisis se concentrará en determinar si la regularización permite controlar el sobreajuste cuando aumenta la complejidad del modelo.

No se asumirá previamente que el modelo regularizado permitirá utilizar un grado mayor. Esta conclusión se establecerá únicamente a partir de los resultados obtenidos.

In [ ]:
# TODO 5
# - Construir el pipeline que combina expansión polinomial y estimador regularizado.
# - Definir la rejilla de tres ejes y estimar el número de ajustes.
# - Ejecutar la búsqueda (en rejilla o aleatoria, justificando la elección).
# - Reportar mejores hiperparámetros y las tres métricas con media y desviación.
# - Comparar el grado óptimo obtenido aquí con el de la sección 2.

## 6. Actividad 5. Comparación y selección del mejor modelo

Una vez terminadas las búsquedas de hiperparámetros, se consolidarán los resultados de validación cruzada de los modelos construidos.

La comparación incluirá:

| Modelo | Mejores hiperparámetros | RMSE medio CV | Desv. RMSE | MAE medio CV | R² medio CV | Complejidad |
|:---|:---|---:|---:|---:|---:|:---|
| Regresión lineal | | | | | | |
| Regresión polinomial | | | | | | |
| Ridge | | | | | | |
| Lasso | | | | | | |
| Polinomial regularizado | | | | | | |

La elección del modelo final no dependerá únicamente de cuál obtenga el menor error promedio.

Tal como establece el enunciado, se tendrán en cuenta tres aspectos:

1. Desempeño promedio en validación cruzada.
2. Estabilidad, representada mediante la desviación estándar entre pliegues.
3. Complejidad del modelo.

La selección se realizará antes de utilizar el conjunto de test.

Al finalizar esta sección se dejará explícitamente identificado el modelo seleccionado y sus mejores hiperparámetros.

In [ ]:
# TODO 6
# - Construir el DataFrame comparativo con una fila por modelo.
# - Ordenar por la métrica principal y mostrarlo.
# - Opcional: gráfico de barras con la media y la barra de error de la desviación.
# - Seleccionar el modelo final y guardarlo en una variable.

### 6.1 Argumentación escrita de la selección

**Qué escribir.** Un párrafo que nombre el modelo elegido, liste sus hiperparámetros ganadores y explique
la elección en las tres dimensiones anteriores. Si el modelo elegido no es el de mejor media, la
explicación de por qué se prefirió la estabilidad es exactamente lo que la rúbrica quiere ver.

## 7. Evaluación final e intervalos de confianza

Una vez terminada la selección de modelos y congeladas todas las decisiones de modelado, se utilizará el conjunto de test.

El conjunto de test no se utilizará para modificar grados, parámetros de regularización, estrategias de escalamiento ni ninguna otra decisión tomada anteriormente.

In [ ]:
# TODO 7
# - Verificar que el modelo final esté ajustado sobre todo el entrenamiento.
# - Predecir sobre test y calcular RMSE, MAE y R2 puntuales.
# - Implementar el bucle de bootstrapping con al menos 500 remuestreos sobre índices.
# - Calcular percentiles 2.5 y 97.5 para cada métrica.
# - Graficar los histogramas de las métricas remuestreadas con el intervalo marcado.
# - Graficar la distribución de residuales y calcular el cociente RMSE/MAE.

### 7.1 Evaluación final sobre el conjunto de test

Los modelos finales obtenidos en las actividades anteriores se evaluarán sobre el conjunto de test utilizando:

- RMSE.
- MAE.
- R².

Esta evaluación se realizará únicamente después de haber terminado la selección de hiperparámetros mediante validación cruzada.

Los resultados se consolidarán en una tabla:

| Modelo | RMSE test | MAE test | R² test |
|:---|---:|---:|---:|
| Regresión lineal | | | |
| Regresión polinomial | | | |
| Ridge | | | |
| Lasso | | | |
| Polinomial regularizado | | | |

Esta tabla permitirá responder posteriormente si el modelo que presentó el mejor comportamiento promedio en validación cruzada coincide con el modelo que obtiene el mejor resultado en el conjunto de test.

Los resultados de test no se utilizarán para reajustar los modelos.

### 7.2 Intervalos de confianza mediante bootstrapping

El procedimiento de bootstrapping se aplicará al modelo seleccionado previamente en la sección 6.

Se utilizará el conjunto de test y se realizarán al menos 500 remuestreos con reemplazo.

En cada remuestreo se calcularán nuevamente las métricas de desempeño utilizando las observaciones reales y sus predicciones correspondientes.

Se estimarán intervalos de confianza del 95 % para:

- RMSE.
- MAE.
- R².

Los intervalos se obtendrán utilizando los percentiles 2.5 y 97.5 de las distribuciones generadas mediante bootstrap.

También se visualizarán las distribuciones de las métricas remuestreadas y de los errores de predicción.

El objetivo de esta etapa es cuantificar la incertidumbre asociada al desempeño del modelo y analizar su estabilidad.

## 8. Análisis de resultados

Las siguientes preguntas se responderán utilizando exclusivamente la evidencia obtenida en las secciones anteriores.

Cada respuesta deberá apoyarse en las métricas, tablas o gráficas producidas durante el laboratorio.

### 8.1 Análisis cuantitativo

#### a. ¿Cuál modelo obtuvo el mejor desempeño en el conjunto de test?

**Evidencia:** tabla de la sección 7.1.

**Respuesta:**

---

#### b. ¿Coincide el mejor desempeño en test con el mejor promedio en validación cruzada? Si no coincide, ¿cuál puede ser la explicación?

**Evidencia:** tablas de las secciones 6 y 7.1.

**Respuesta:**

---

#### c. ¿El modelo con mejor métrica promedio es necesariamente el más adecuado?

**Evidencia:** promedio y desviación estándar de las métricas de validación cruzada.

**Respuesta:**

---

#### d. Con base en las curvas de validación, ¿cómo cambia el error a medida que aumenta la complejidad? ¿En qué punto se evidencia sobreajuste?

**Evidencia:** curva de validación de la sección 3.

**Respuesta:**

---

#### e. ¿Cómo afecta la regularización la magnitud y estabilidad de los coeficientes?

**Evidencia:** comparación entre regresión lineal, Ridge y Lasso.

**Respuesta:**

---

#### f. ¿Los intervalos de confianza obtenidos mediante bootstrapping sugieren estabilidad o alta variabilidad en el desempeño? ¿Qué implicaciones tiene esto?

**Evidencia:** resultados de la sección 7.2.

**Respuesta:**

### 8.2 Análisis cualitativo

#### a. ¿Qué variables fueron seleccionadas como más relevantes por el modelo Lasso?

**Evidencia:** coeficientes del mejor modelo Lasso.

**Respuesta:**

---

#### b. ¿Qué interpretación práctica tienen los coeficientes del modelo final en el contexto de la estimación de temperatura máxima?

**Evidencia:** coeficientes y representación utilizada por el modelo final.

**Respuesta:**

---

#### c. ¿Existen diferencias relevantes entre el modelo más preciso y el más interpretable?

**Evidencia:** resultados comparativos de desempeño y complejidad.

**Respuesta:**

---

#### d. ¿Qué decisiones estratégicas podría tomar AlpesPlanck a partir de los resultados obtenidos?

**Evidencia:** resultados del modelo final y variables relevantes.

**Respuesta:**

---

#### e. ¿Mayor precisión implica necesariamente mayor valor para la organización?

**Respuesta:**

---

#### f. ¿Un modelo más complejo necesariamente genera mayor valor empresarial?

Considerar interpretabilidad, estabilidad y costo de implementación.

**Respuesta:**

### 8.3 Reflexión conceptual

#### a. ¿Qué relación observas entre complejidad del modelo, capacidad de generalización y estabilidad del desempeño?

**Evidencia:** curvas de validación, resultados de regularización y bootstrap.

**Respuesta:**

---

#### b. ¿Qué fuentes de sesgo podrían estar presentes en los datos o en el proceso de modelado?

**Respuesta:**

---

#### c. Si el tamaño de muestra fuera mayor, ¿esperarías cambios en la estabilidad de los modelos?

**Respuesta:**

---
## 9. Actividad 7. Video explicativo
**Rúbrica: criterio 7, 5 %. Máximo 3 minutos.**

**Audiencia.** El ingeniero de machine learning que lidera el grupo de AlpesPlanck. Es una audiencia
técnica: no hay que explicar qué es la validación cruzada, hay que explicar qué decidiste y por qué.

**Estructura sugerida para tres minutos.**

1. Punto de partida y objetivo, con el resultado del Laboratorio 1 como referencia (unos 20 segundos).
2. Los modelos comparados y el criterio de comparación (40 segundos).
3. El hallazgo sobre complejidad y sobreajuste, mostrando la curva de validación (40 segundos).
4. El modelo seleccionado, sus hiperparámetros y por qué se eligió, incluyendo la estabilidad (40 segundos).
5. El intervalo de confianza y qué significa para el uso de la predicción (30 segundos).
6. Limitaciones y siguiente paso recomendado (10 segundos).

**Error frecuente.** Gastar dos minutos describiendo el código. El líder técnico quiere las decisiones y
la evidencia, no un recorrido por las celdas.

---
## 10. Uso de herramientas de IA generativa
**Rúbrica: criterio 8, 5 %. La sección debe llevar exactamente este título.**

Según las reglas del curso, esta sección debe contener cuatro componentes. Si falta cualquiera de ellos,
el criterio se pierde completo, que es una forma barata de perder 5 %.

**1. Declaración del uso.** Nombre de la herramienta y tipo de uso: ayuda conceptual, generación inicial
de código, explicación teórica, depuración, redacción, u otro.

**2. Prompts utilizados.** De forma textual o resumida. No hacen falta las interacciones menores, pero sí
las que influyeron directamente en el resultado entregado.

**3. Análisis crítico del resultado.** Hay que responder al menos dos de estas preguntas:

- ¿Qué partes del contenido generado fueron correctas y útiles?
- ¿Qué errores, imprecisiones o limitaciones se identificaron?
- ¿Qué decisiones técnicas fueron modificadas respecto a la respuesta de la IA y por qué?
- ¿Qué conceptos del curso permitieron evaluar o mejorar la respuesta generada?

**4. Aportes propios.** Qué fue desarrollado, modificado o decidido por ustedes; qué ajustes se hicieron
sobre el código o la explicación original; y qué aprendizajes se obtuvieron del proceso.

**Nota del esqueleto.** Si usaste este esqueleto, es un uso de IA generativa y debe declararse. El
análisis crítico correspondiente puede apoyarse en las decisiones que el esqueleto dejó abiertas de forma
deliberada y que ustedes tuvieron que resolver, por ejemplo el tipo de partición frente a la estructura
temporal de los datos, o la elección entre Ridge y Lasso para el modelo polinomial regularizado.

---
## 11. Lista de verificación antes de entregar

**Contenido técnico**

- [ ] Todas las celdas ejecutadas en orden desde un kernel limpio, con las salidas visibles.
- [ ] Semilla fija en particiones, validación cruzada y bootstrapping.
- [ ] El mismo objeto de validación cruzada en las cuatro búsquedas.
- [ ] Todo el preprocesamiento dentro de los pipelines, sin transformaciones previas a la partición.
- [ ] RMSE, MAE y R² reportados con el signo correcto en todos los modelos.
- [ ] El conjunto de test no se utilizó durante la búsqueda de hiperparámetros ni durante la selección del modelo.
- [ ] Todas las decisiones de modelado quedaron congeladas antes de evaluar sobre test.
- [ ] Los modelos finales fueron evaluados sobre test únicamente después de finalizar la validación cruzada.
- [ ] El bootstrapping se realizó sobre el modelo seleccionado previamente.
- [ ] Se realizaron al menos 500 remuestreos para los intervalos de confianza.
- [ ] Al menos 500 remuestreos en el bootstrapping.
- [ ] Sin advertencias de convergencia sin resolver ni documentar.

**Documentación**

- [ ] Cada decisión metodológica justificada por escrito, no solo implementada.
- [ ] Las quince preguntas del análisis de resultados respondidas y ancladas a números o figuras propios.
- [ ] Todas las figuras con título, ejes rotulados y leyenda.
- [ ] Notas del esqueleto eliminadas.

**Entrega**

- [ ] Archivos `.ipynb` y `.html` con los nombres de los estudiantes.
- [ ] Video de máximo 3 minutos.
- [ ] Sección "Uso de herramientas de IA generativa" con los cuatro componentes.
- [ ] Los dos integrantes que presentan registrados en el grupo GL2 para habilitar el enlace.
- [ ] Entrega antes del 14 de septiembre a las 20:00. Entre esa hora y las 2:00 del 15 de septiembre la
      entrega se califica sobre 3.5; después de ese momento, sobre 0.